# Recreate image with circles

## Import modules

In [2]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
# from pathlib import Path
from datetime import date
# from dataclasses import dataclass, field
import itertools


# import 3rd-party modules
import cv2
import numpy as np
from numba import njit
import ray
# from pygifsicle import optimize


# import local modules
from utils.renderer.giffer import create_gif
# from utils.renderer.videographer import create_video
from utils.project_manager import Project
from utils.renderer.resizer import get_interpolation
# from utils.renderer.resizer import resize_with_pad, resize_with_crop

## Set up project

In [19]:
# name out img dirs
out_img_dir_list = ["circles_emin"]

# create project
project = Project(project_dir="assets/images/mosaic/circle", out_img_dir_list=out_img_dir_list)

## Define functions and classes

In [15]:
# shutdown ray
ray.shutdown()

# start ray
ray.init(log_to_driver=True)

@ray.remote
def create_mosaic_with_circles(dest_img, nb_rows, nb_cols, col_range=None):
    """
    Function to recreate an image with circles

    Arguments
    * dest_img: destination image, i.e. image to recreate with circles
    """

    # get cell height & width
    cell_height, cell_width = dest_img.shape[0]//nb_rows, dest_img.shape[1]//nb_cols

    # get list of grid positions
    if col_range is not None:
        grid_positions = [(grid_y, grid_x) for grid_y in range(*col_range) for grid_x in range(nb_rows)]
        out_img = np.zeros_like(dest_img[:,col_range[0]:col_range[1]])
    else:
        grid_positions = [(grid_y, grid_x) for grid_y in range(nb_cols) for grid_x in range(nb_rows)]
        out_img = np.zeros_like(dest_img)

    # get index range of grid positions
    grid_positions_idxs = np.arange(len(grid_positions))
    # # choose random seed to recreate same shuffle or change it to see if you get better results
    # np.random.seed(1111)
    # shuffle grid positions (otherwise the first grids from top will get the best matching images)
    np.random.shuffle(grid_positions_idxs)

    # iterate over each grid position
    for i in grid_positions_idxs:

        grid_y, grid_x = grid_positions[i]
        
        # get region of interest in img to recreate
        y = grid_y * cell_height
        x = grid_x * cell_width
        dest_roi = dest_img[y:y+cell_height, x:x+cell_width]

        # get mean of roi
        dest_roi_mean = np.mean(dest_roi, dtype=np.uint8).tolist()

        # get center coordinates
        center_y, center_x = (y+cell_height)//2, (x+cell_width)//2

        # Radius of circle
        radius = max(cell_height, cell_width)

        # fill circle with color
        thickness = -1

        # draw circle on roi
        out_img = cv2.circle(out_img, (center_x, center_y), radius, dest_roi_mean, thickness)

    return out_img

## Define constants & variables

In [22]:
# set grid caracteristics
NB_ROWS = 200
NB_COLS = 200

# choose number of tasks
NB_TASKS = 4

# set dest image path (i.e path of image to recreate)
dest_img_path = "/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/emin_shut/emin_celeb.jpeg"

## Read image

In [23]:
# get dest image
dest_img = cv2.imread(dest_img_path)

## Recreate image

In [17]:
# start tasks in parallel
result_ids = []

for i in range(0, NB_COLS, NB_COLS//NB_TASKS): # set grid range for each worker

    # recreate given image with images mosaic
    result_ids.append(create_mosaic_with_circles.remote(dest_img=dest_img, nb_rows=NB_ROWS, nb_cols=NB_COLS, col_range=(i, i+NB_COLS//NB_TASKS)))

# wait for the tasks to complete and retrieve the results
results = ray.get(result_ids)

out_img = cv2.hconcat(results)

# set output image directory
out_img_dir = "circles"

# get current date
today = date.today().strftime("%Y%m%d")

# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"mosaic_{NB_ROWS*NB_ROWS}_{today}.jpg")

# save output image & metadata
cv2.imwrite(out_img_path, out_img)

True

In [85]:
def find_nearest_multiple(x, base):
    """
    Function to find the nearest multiple of base given an integer
    """
    return base * round(x/base)

In [89]:
def create_mosaic_with_circles(dest_img, nb_rows, nb_cols, col_range=None):
    """
    Function to recreate an image with circles

    Arguments
    * dest_img: destination image, i.e. image to recreate with circles
    """

    # get img height & width
    dest_img_height, dest_img_width = dest_img.shape[:2]

    # get best number of rows and cols to cover the whole img
    out_img_height = find_nearest_multiple(dest_img_height, nb_rows)
    out_img_width = find_nearest_multiple(dest_img_width, nb_cols)

    # resize image to recreate so that the grid can cover the whole image
    # get interpolation
    interpolation = get_interpolation(src_img_shape=(dest_img_height, dest_img_width), out_img_shape=(out_img_height, out_img_width))
    dest_img = cv2.resize(dest_img, (out_img_width, out_img_height), interpolation=interpolation)

    # get cell height & width
    cell_height, cell_width = out_img_height//nb_rows, out_img_width//nb_cols

    # get list of grid positions
    if col_range is not None:
        grid_positions = [(grid_y, grid_x) for grid_x in range(*col_range) for grid_y in range(nb_rows)]
        out_img = np.zeros_like(dest_img[:,col_range[0]:col_range[1]])
    else:
        grid_positions = [(grid_y, grid_x) for grid_x in range(nb_cols) for grid_y in range(nb_rows)]
        out_img = np.zeros_like(dest_img)

    # get index range of grid positions
    grid_positions_idxs = np.arange(len(grid_positions))
    # # choose random seed to recreate same shuffle or change it to see if you get better results
    # np.random.seed(1111)
    # shuffle grid positions (otherwise the first grids from top will get the best matching images)
    # np.random.shuffle(grid_positions_idxs)

    # iterate over each grid position
    for i in grid_positions_idxs:

        grid_y, grid_x = grid_positions[i]
        
        # get region of interest in img to recreate
        y = grid_y * cell_height
        x = grid_x * cell_width
        dest_roi = dest_img[y:y+cell_height, x:x+cell_width]
        
        # get mean of roi
        dest_roi_mean = cv2.mean(dest_roi)[:-1]

        # get center coordinates
        center_y, center_x = (y+cell_height//2), (x+cell_width//2)

        # Radius of circle
        # radius = min(cell_height//2, cell_width//2)
        min_cell_side = min(cell_height//2, cell_width//2)
        radius = np.random.randint(min_cell_side, min_cell_side*8)

        # min_cell_side = min(cell_height//2, cell_width//2)
        # max_cell_side = max(cell_height//2, cell_width//2)
        
        # # (major axis length, minor axis length)
        # axesLength = (max_cell_side, min_cell_side)
        
        # # angle = 0
        # angle = np.random.randint(0, 360)
        
        # startAngle = np.random.randint(0, 360)
        
        # endAngle = np.random.randint(startAngle, 360)
        
        # fill circle with color
        thickness = -1

        # # draw ellipse
        # out_img = cv2.ellipse(out_img, (center_x, center_y), axesLength,
        #         angle, startAngle, endAngle, dest_roi_mean, thickness)

        # draw circle on roi
        out_img = cv2.circle(out_img, (center_x, center_y), radius, dest_roi_mean, thickness)

    return out_img

In [90]:
for i in range(60):

    out_img = create_mosaic_with_circles(dest_img=dest_img, nb_rows=100, nb_cols=100)
    # set output image directory
    out_img_dir = "circles_emin"

    # get current date
    today = date.today().strftime("%Y%m%d")

    # set output image path
    out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"mosaic_{NB_ROWS*NB_ROWS}_{today}_{i}.jpg")

    # save output image & metadata
    cv2.imwrite(out_img_path, out_img)

In [91]:
# get img height & width
dest_img_height, dest_img_width = dest_img.shape[:2]

# set gif path
gif_path = project.project_dir / f"random_{out_img_dir}.gif"

# create gif
create_gif(img_dir=project.out_img_dir_dict[out_img_dir], out_path=gif_path, out_img_shape=(int(dest_img_height),int(dest_img_width)),
sort_img_list=True, duplicate_start_img_amount=0, duplicate_end_img_amount=0)